In [4]:
from insightface.app import FaceAnalysis
import numpy as np

app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0)  # ctx_id=0 là GPU, -1 là CPU

img1 = cv2.imread("person1.jpg")
img2 = cv2.imread("person2.jpg")

faces1 = app.get(img1)
faces2 = app.get(img2)

# So sánh embedding
emb1 = faces1[0].normed_embedding
emb2 = faces2[0].normed_embedding
similarity = np.dot(emb1, emb2)  # cosine similarity (đã normalize)
print("Same person:", similarity > 0.4)

ImportError: cannot import name 'builder' from 'google.protobuf.internal' (c:\Users\Admin\Desktop\Project_DAT301m\.venv\lib\site-packages\google\protobuf\internal\__init__.py)

In [ ]:
from deepface import DeepFace
import numpy as np

# Verify 2 ảnh
result = DeepFace.verify(
    img1_path=r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg",      # ảnh chứng minh
    img2_path=r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test6.jpg",    # ảnh chụp thực tế
    model_name="ArcFace",      # model mạnh nhất
    detector_backend="mtcnn",  # detect + align tốt nhất
    distance_metric="cosine"
)

print("Cùng người:", result["verified"])
print("Khoảng cách:", result["distance"])   # < 0.68 → same person
print("Threshold:", result["threshold"])

Cùng người: True
Khoảng cách: 0.649265
Threshold: 0.68


In [11]:
from deepface import DeepFace
import numpy as np
import time

IMG1 = r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg"
IMG2 = r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test1.jpg"

models_to_try = ["ArcFace", "Facenet512", "SFace", "VGG-Face"]
detectors = ["retinaface", "mtcnn"]
metrics = ["cosine", "euclidean_l2"]

print(f"{'Model':<15} {'Detector':<12} {'Metric':<14} {'Distance':<10} {'Threshold':<10} {'Verified':<10} {'Time':<8}")
print("=" * 80)

for model in models_to_try:
    for det in detectors:
        for met in metrics:
            try:
                start = time.time()
                r = DeepFace.verify(IMG1, IMG2, model_name=model,
                                    detector_backend=det, distance_metric=met,
                                    enforce_detection=False)
                elapsed = time.time() - start
                print(f"{model:<15} {det:<12} {met:<14} {r['distance']:<10.4f} {r['threshold']:<10.4f} {str(r['verified']):<10} {elapsed:<8.2f}s")
            except Exception as e:
                print(f"{model:<15} {det:<12} {met:<14} ERROR: {str(e)[:50]}")

# --- Ensemble voting ---
print("\n" + "=" * 80)
print("ENSEMBLE VOTING (cosine + retinaface)")
votes = []
for model in models_to_try:
    try:
        r = DeepFace.verify(IMG1, IMG2, model_name=model,
                            detector_backend="retinaface", distance_metric="cosine",
                            enforce_detection=False)
        votes.append(r["verified"])
        print(f"  {model:<15} -> {'SAME' if r['verified'] else 'DIFF'} (dist={r['distance']:.4f}, thresh={r['threshold']:.4f})")
    except Exception as e:
        print(f"  {model:<15} -> ERROR: {str(e)[:40]}")

yes = sum(votes)
total = len(votes)
print(f"\n  Kết luận: {yes}/{total} models đồng ý -> {'SAME PERSON' if yes > total//2 else 'DIFFERENT'}")

# --- Cosine similarity từ embedding (dùng Facenet512) ---
print("\n" + "=" * 80)
print("COSINE SIMILARITY từ embedding (Facenet512 + retinaface)")
try:
    emb1 = DeepFace.represent(IMG1, model_name="Facenet512", detector_backend="retinaface", enforce_detection=False)[0]["embedding"]
    emb2 = DeepFace.represent(IMG2, model_name="Facenet512", detector_backend="retinaface", enforce_detection=False)[0]["embedding"]
    emb1 = np.array(emb1) / np.linalg.norm(emb1)
    emb2 = np.array(emb2) / np.linalg.norm(emb2)
    similarity = float(np.dot(emb1, emb2))
    print(f"  Cosine similarity: {similarity:.6f}")
    print(f"  Cosine distance (1 - sim): {1 - similarity:.6f}")
    print(f"  Kết luận với ngưỡng 0.30: {'SAME' if similarity > 0.30 else 'DIFF'}")
except Exception as e:
    print(f"  ERROR: {e}")

Model           Detector     Metric         Distance   Threshold  Verified   Time    
ArcFace         retinaface   cosine         0.6767     0.6800     True       0.50    s
ArcFace         retinaface   euclidean_l2   1.1633     1.1300     False      0.39    s
ArcFace         mtcnn        cosine         0.7707     0.6800     False      3.90    s
ArcFace         mtcnn        euclidean_l2   1.2415     1.1300     False      3.88    s
Facenet512      retinaface   cosine         0.1721     0.3000     True       0.67    s
Facenet512      retinaface   euclidean_l2   0.5868     1.0400     True       0.46    s
Facenet512      mtcnn        cosine         0.1689     0.3000     True       3.96    s
Facenet512      mtcnn        euclidean_l2   0.5811     1.0400     True       3.96    s
SFace           retinaface   cosine         0.6700     0.5930     False      0.35    s
SFace           retinaface   euclidean_l2   1.1576     1.0550     False      0.33    s
SFace           mtcnn        cosine         